# Tabela de Usuários: users
A tabela de usuários apresenta usuários unicos do ambiente, desta forma sem apresentar redundancia de registros de pessoas que utilizam o ambiente Databricks
- A tabela possui como chave primária o email do usuário, informação extraida da tabela system.access.audit
- Formato Struct das informações customizaveis do respectivo usuário.

## Pipeline: users
Essa pipeline é referente a pipeline de ingestão de dados na tabela de usuários no ambiente Databricks, desta forma uma relação dos novos usuários da plataforma, possuindo como caracteristica inserir o usuário na lista apenas se não existir o registro dele, além de preservar o restante das informações. Os passos executados são os seguintes
- Exclui a tabela, se ela existir;
- Realiza a criação da tabela;
- Atribui ao campo "email" a caracteristica de chave primária;
- Realiza o streaming de dados com a origem da informação;
- A insersão de dados acontece sob duas condições separadas nos blocos:
  - Caso o usuário não exista na tabela, ele será incluido;
  - Caso o usuário tenha data inicial diferente da data final, desta forma companhando a ultima vez que ele acesso ao ambiente.

In [0]:
%sql
DROP TABLE IF EXISTS data_governance.users;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS data_governance.users(
  email STRING NOT NULL
  , user_info STRUCT <
  username: STRING
  , company: STRING
  , area: STRING
  , team: STRING
  , first_access_dt: DATE
  , last_access_dt DATE
  , status: STRING
  >
);

In [0]:
%sql
ALTER TABLE data_governance.users ADD CONSTRAINT pk_email PRIMARY KEY(email);
    
ALTER TABLE data_governance.users ADD CONSTRAINT chk_status CHECK (user_info.status IN ('active','inactive'))


In [0]:
%sql
COMMENT ON TABLE data_governance.users IS '## Content
This table contains each user information, like email, full name, company, area, team, first and last access date and status
### Update
This table updates every day 0:00 AM (UTC-03:00)';
COMMENT ON COLUMN data_governance.users.email IS 'Unique email that identifies each user';
COMMENT ON COLUMN data_governance.users.user_info IS '# Struct field that has details of each user:
- username: Full name of the user;
- company: Company that the user belong;
- area: Area that the user belongs;
- tema: Team that the user belongs;
- first_access_date: First date that the user accessed the platform;
- last_access_date: Last date that the user accessed the platform;
- status: Status of the user if active or inactive;';

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE data_governance.users(
    email STRING
    , user_info STRUCT <
    username: STRING
    , company: STRING
    , area: STRING
    , team: STRING
    , first_access_dt: DATE
    , last_access_dt: DATE
    , status: STRING
    >
) AS 
SELECT DISTINCT
  user_identity.email as email
  , MIN(event_date) as first_access_dt
  , MAX(event_date) as last_acess_dt
FROM STREAM(system.access.audit)
WHERE user_identity.email RLIKE '@';

In [0]:
%sql
INSERT INTO data_governance.users(email,user_info)
SELECT DISTINCT
  user_identity.email as email
  , named_struct('username',null,'company',null,'area',null,'team',null, 'first_access_dt', MIN(event_date),'last_access_dt', MAX(event_date),'status','active') as user_info
FROM system.access.audit
WHERE user_identity.email RLIKE '@'
  AND NOT EXISTS (
    SELECT
      email
      , user_info.username
      , user_info.company
      , user_info.area
      , user_info.team
      , user_info.first_access_dt
      , user_info.last_access_dt
      , user_info.status
    FROM data_governance.users u 
    WHERE (u.email = user_identity.email) OR (user_info.first_access_dt <> user_info.last_access_dt))
  GROUP BY ALL;